In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 

In [2]:
models = pd.read_parquet('./data/models.parquet')

models['include_ensemble'] = models['designation'] == 'primary'

models.head()

,model,designation,include_ensemble
0,KITmetricslab-select_ensemble,primary,True
1,CovidAnalytics-DELPHI,primary,True
2,UMass-MechBayes,primary,True
3,UT-Osiris,primary,True
4,USACE-ERDC_SEIR,primary,True


In [3]:
models.loc[models.include_ensemble == True].model.unique()


array(['KITmetricslab-select_ensemble', 'CovidAnalytics-DELPHI',
       'UMass-MechBayes', 'UT-Osiris', 'USACE-ERDC_SEIR', 'OHT-nbx',
       'IHME-CurveFit', 'Imperial-ensemble2', 'UA-EpiCovDA', 'STH-3PU',
       'JHUAPL-Bucky', 'MOBS-GLEAM_COVID', 'GT_CHHS-COVID19',
       'Auquan-SEIR', 'UCF-AEM', 'OneQuietNight-ML',
       'UCM_MESALab-FoGSEIR', 'CovidActNow-SEIR_CAN', 'IBF-TimeSeries',
       'CDDEP-ABM', 'MUNI-ARIMA', 'LANL-GrowthRate', 'GT-DeepCOVID',
       'AMM-EpiInvert', 'ISUandPKU-vSEIdR', 'SWC-TerminusCM',
       'NotreDame-mobility', 'Yu_Group-CLEP', 'PR_UMD-CF_RepTiLe',
       'CEID-Walk', 'IQVIA_ACOE-STAN', 'COVIDhub-ensemble',
       'epiforecasts-ensemble1', 'JHUAPLTDWG-ICATTML',
       'UChicagoCHATTOPADHYAY-UnIT', 'FRBSF_Wilson-Econometric',
       'HKUST-DNN', 'CEPH-Rtrend_covid', 'QJHong-Encounter',
       'MITCovAlliance-SIR', 'Quantori-Multiagents', 'TTU-squider',
       'USF-STPM', 'YYG-ParamSearch', 'PSI-DRAFT', 'CUBoulder-COVIDLSTM',
       'SDSC_ISG-TrendMode

In [4]:
df_preds = pd.read_parquet('./data/fullforecasts.parquet')

df_preds.head()

,model,forecast_date,location,horizon,temporal_resolution,target_variable,target_end_date,type,quantile,value,location_name,population,geo_type,geo_value,abbreviation,full_location_name
0,BPagano-RtDriven,2020-10-18,US,1,wk,inc case,2020-10-24,point,NaN,419616.58481,United States,332875137.0,state,us,US,United States
1,BPagano-RtDriven,2020-10-18,US,1,wk,inc case,2020-10-24,quantile,0.025,252157.46981,United States,332875137.0,state,us,US,United States
2,BPagano-RtDriven,2020-10-18,US,1,wk,inc case,2020-10-24,quantile,0.100,309027.43446,United States,332875137.0,state,us,US,United States
3,BPagano-RtDriven,2020-10-18,US,1,wk,inc case,2020-10-24,quantile,0.250,360710.91108,United States,332875137.0,state,us,US,United States
4,BPagano-RtDriven,2020-10-18,US,1,wk,inc case,2020-10-24,quantile,0.500,418324.95041,United States,332875137.0,state,us,US,United States


In [6]:
df_preds['quantile'].unique()

array([  nan, 0.025, 0.1  , 0.25 , 0.5  , 0.75 , 0.9  , 0.975])

In [8]:
df_preds_final = pd.concat([df_preds.loc[(df_preds.type == 'quantile') & (df_preds['quantile'] == 0.025)][['model', 'forecast_date', 'horizon',
      'target_end_date', 'value']].rename(columns = {'value': 'lower_95', 'forecast_date':'epiweek', 'target_end_date':'date'}).set_index(['model', 'epiweek', 'horizon', 'date']),

    df_preds.loc[(df_preds.type == 'quantile') & (df_preds['quantile'] == 0.1)][['model', 'forecast_date', 'horizon',
      'target_end_date', 'value']].rename(columns = {'value': 'lower_80', 'forecast_date':'epiweek', 'target_end_date':'date'}).set_index(['model', 'epiweek', 'horizon', 'date']),

    df_preds.loc[(df_preds.type == 'quantile') & (df_preds['quantile'] == 0.25)][['model', 'forecast_date', 'horizon',
      'target_end_date', 'value']].rename(columns = {'value': 'lower_50', 'forecast_date':'epiweek', 'target_end_date':'date'}).set_index(['model', 'epiweek', 'horizon', 'date']),
                       
           df_preds.loc[(df_preds.type == 'quantile') & (df_preds['quantile'] == 0.5)][['model', 'forecast_date', 'horizon',
      'target_end_date', 'value']].rename(columns = {'value': 'pred', 'forecast_date':'epiweek', 'target_end_date':'date'}).set_index(['model', 'epiweek', 'horizon', 'date']), 
           
        df_preds.loc[(df_preds.type == 'quantile') & (df_preds['quantile'] == 0.75)][['model', 'forecast_date', 'horizon',
      'target_end_date', 'value']].rename(columns = {'value': 'upper_50', 'forecast_date':'epiweek', 'target_end_date':'date'}).set_index(['model', 'epiweek', 'horizon', 'date']),

     df_preds.loc[(df_preds.type == 'quantile') & (df_preds['quantile'] == 0.9)][['model', 'forecast_date', 'horizon',
      'target_end_date', 'value']].rename(columns = {'value': 'upper_80', 'forecast_date':'epiweek', 'target_end_date':'date'}).set_index(['model', 'epiweek', 'horizon', 'date']),

    df_preds.loc[(df_preds.type == 'quantile') & (df_preds['quantile'] == 0.975)][['model', 'forecast_date', 'horizon',
      'target_end_date', 'value']].rename(columns = {'value': 'upper_95', 'forecast_date':'epiweek', 'target_end_date':'date'}).set_index(['model', 'epiweek', 'horizon', 'date'])], axis =1).reset_index()

df_preds_final = df_preds_final.loc[df_preds_final.model.isin(models.loc[models.designation =='primary'].model.unique())]

df_preds_final = df_preds_final.loc[~df_preds_final.model.isin(['OliverWyman-Navigator'])]

df_preds_final = df_preds_final.rename(columns = {'model':'model_id'})

df_preds_final = df_preds_final.loc[df_preds_final.upper_95 > 10]

df_preds_final = df_preds_final.dropna()

df_preds_final.head()

,model_id,epiweek,horizon,date,lower_95,lower_80,lower_50,pred,upper_50,upper_80,upper_95
0,BPagano-RtDriven,2020-10-18,1,2020-10-24,252157.46981,309027.43446,360710.91108,418324.95041,475938.98974,527622.46637,584492.43102
1,BPagano-RtDriven,2020-10-18,2,2020-10-31,250557.34760,325076.22060,392799.04150,468292.90380,543786.76609,611509.58699,686028.46000
2,BPagano-RtDriven,2020-10-18,3,2020-11-07,236909.92449,332846.68480,420034.10156,517226.07830,614418.05504,701605.47179,797542.23210
3,BPagano-RtDriven,2020-10-18,4,2020-11-14,213876.19851,330803.52595,437067.18958,555524.36963,673981.54967,780245.21330,897172.54074
4,BPagano-RtDriven,2020-10-25,1,2020-10-31,299044.13571,364922.81525,424793.42033,491534.04269,558274.66504,618145.27012,684023.94966


In [14]:
df_preds_final.loc[(df_preds_final.date == pd.to_datetime('2020-10-24').date()) & (df_preds_final.horizon == '1')].model_id.value_counts()

model_id
CU-select                2
LANL-GrowthRate          2
BPagano-RtDriven         1
IowaStateLW-STEM         1
RobertWalraven-ESG       1
LNQ-ens1                 1
Karlen-pypm              1
JHU_IDD-CovidSP          1
JHUAPL-Bucky             1
DDS-NBDS                 1
CEID-Walk                1
CovidAnalytics-DELPHI    1
Covid19Sim-Simulator     1
Columbia_UNC-SurvCon     1
COVIDhub_CDC-ensemble    1
COVIDhub-ensemble        1
UCLA-SuEIR               1
Name: count, dtype: int64

In [9]:
df_ens_covidhub = df_preds_final.groupby(['date', 'epiweek', 'horizon'])[['lower_95','lower_80','lower_50', 'pred', 'upper_50', 'upper_80', 'upper_95']].median().reset_index()

df_ens_covidhub.head()

,date,epiweek,horizon,lower_95,lower_80,lower_50,pred,upper_50,upper_80,upper_95
0,2020-07-11,2020-07-03,1,165914.725884,223544.000186,279688.142053,354132.136774,407960.037553,440312.615941,547593.214527
1,2020-07-11,2020-07-05,1,368784.000000,380337.000000,390252.000000,402431.242400,408263.669948,413187.754085,440146.000000
2,2020-07-11,2020-07-06,1,197791.197700,222450.468727,248754.288674,277726.697072,295434.507048,308384.971600,341125.871600
3,2020-07-18,2020-07-03,2,104593.590239,160360.880005,224530.439371,311021.842637,412816.743816,495608.244782,653208.649786
4,2020-07-18,2020-07-05,2,359542.000000,375999.000000,392876.000000,416187.000000,437465.488413,461124.000000,480405.000000


In [10]:
df_ens_covidhub.to_csv('covidhub_untrained.csv', index = False)